In [1]:
# ===== STEP 1. 상품 클래스 =====
class Product:
    _next_id = 1                        # 클래스 변수: 다음에 줄 번호표

    def __init__(self, name, price, stock):
        self.id = Product._next_id      # 번호표 받기
        Product._next_id += 1           # 다음 사람을 위해 번호표 +1
        self.name = name
        self.price = price
        self._stock = stock             # _ : 직접 수정 금지 표시

    def get_stock(self):                # 재고 "읽기"
        return self._stock

    def is_sold_out(self):              # 품절인가?
        return self._stock == 0

    def reduce_stock(self, qty):        # 재고 "줄이기" (유일한 변경 통로)
        self._stock -= qty

    def __str__(self):
        if self.is_sold_out():
            stock_text = "품절"
        else:
            stock_text = f"재고 {self._stock}개"
        return f"{self.id}. {self.name:<10} {self.price:>8,}원  ({stock_text})"









# ===== STEP 2. 할인 쿠폰 클래스 =====
class Discount:                          # 부모: 쿠폰 없음
    def __init__(self, name="쿠폰 없음"):
        self.name = name

    def discount_amount(self, total):    # total = 상품 합계
        return 0                         # 항상 0원


class RateDiscount(Discount):            # 자식 1: 비율 할인
    def __init__(self, rate, max_amount):
        super().__init__(f"{round(rate * 100)}% 할인 (최대 {max_amount:,}원)")
        self.rate = rate
        self.max_amount = max_amount

    def discount_amount(self, total):    # 오버라이딩
        amount = int(total * self.rate)  # int(): 원 단위 미만 버림
        return min(amount, self.max_amount)   # 최대 금액을 넘지 않게


class FixedDiscount(Discount):           # 자식 2: 정액 할인
    def __init__(self, amount, min_total):
        super().__init__(f"{amount:,}원 할인 ({min_total:,}원 이상 구매 시)")
        self.amount = amount
        self.min_total = min_total

    def discount_amount(self, total):    # 오버라이딩
        if total >= self.min_total:
            return self.amount
        return 0







# ===== STEP 3. 장바구니 클래스 =====
class Cart:
    def __init__(self):
        self.items = {}                  # {Product: 수량}
        self.coupon = Discount()         # 처음엔 "쿠폰 없음"

    # ---- 담기 ----
    def add(self, product, qty):
        if product.is_sold_out():
            print("품절된 상품입니다.")
            return

        in_cart = self.items.get(product, 0)        # 이미 담은 수량 (없으면 0)
        available = product.get_stock() - in_cart   # 더 담을 수 있는 수량
        if qty > available:
            print(f"재고가 부족합니다. (담을 수 있는 수량: {available}개)")
            return

        self.items[product] = in_cart + qty         # 이미 있으면 더하기
        print(f"{product.name} {qty}개를 담았습니다.")

    # ---- 빼기 ----
    def remove(self, product):
        if product in self.items:
            del self.items[product]
            print(f"{product.name}을(를) 장바구니에서 뺐습니다.")
        else:
            print("장바구니에 없는 상품입니다.")

    # ---- 쿠폰 적용 ----
    def set_coupon(self, coupon):
        self.coupon = coupon
        print(f"[{coupon.name}] 쿠폰을 적용했습니다.")

    # ---- 금액 계산 (발제문 3-4의 ①~④) ----
    def subtotal(self):                                  # ① 상품 합계
        return sum(p.price * q for p, q in self.items.items())

    def discount(self):                                  # ② 할인 금액
        return self.coupon.discount_amount(self.subtotal())

    def shipping(self):                                  # ③ 배송비
        if not self.items:                               # 비었으면 0원
            return 0
        if self.subtotal() - self.discount() >= 30000:
            return 0
        return 3000

    def total(self):                                     # ④ 결제 금액
        return self.subtotal() - self.discount() + self.shipping()

    # ---- 출력 (장바구니/영수증 공용) ----
    def print_summary(self):
        print("-" * 40)
        for p, q in self.items.items():
            print(f"{p.name:<10} x {q:<3} {p.price * q:>12,}원")
        print("-" * 40)
        print(f"{'상품 합계':<14}{self.subtotal():>14,}원")
        print(f"{'할인':<14}{-self.discount():>14,}원  [{self.coupon.name}]")
        print(f"{'배송비':<14}{self.shipping():>14,}원")
        print(f"{'결제 금액':<14}{self.total():>14,}원")
        print("-" * 40)

    # ---- 결제 ----
    def checkout(self):
        if not self.items:
            print("장바구니가 비어 있습니다.")
            return
        print("\n========== 영수증 ==========")
        self.print_summary()
        for p, q in self.items.items():      # 재고 감소
            p.reduce_stock(q)
        self.items.clear()                   # 장바구니 비우기
        self.coupon = Discount()             # 쿠폰 초기화
        print("결제가 완료되었습니다. 감사합니다!")








# ===== STEP 4. 메인 프로그램 =====
def ask_number(message):
    """숫자를 입력받아 int로 반환. 숫자가 아니면 None"""
    text = input(message)
    if not text.isdecimal():
        print("숫자만 입력해주세요.")
        return None
    return int(text)


def find_product(products, number):
    """번호로 상품 찾기. 없으면 None"""
    for p in products:
        if p.id == number:
            return p
    print("없는 상품 번호입니다.")
    return None







# 상품 5개 (번호는 자동 부여)
products = [
    Product("무선 이어폰", 89000, 3),
    Product("텀블러", 15000, 10),
    Product("보조배터리", 32000, 5),
    Product("마우스패드", 9000, 0),
    Product("USB 허브", 18000, 4),
]

# 쿠폰 3개
coupons = [
    Discount(),
    RateDiscount(0.1, 5000),
    FixedDiscount(3000, 20000),
]

cart = Cart()

while True:
    print("-" * 46)
    print("1.상품목록 2.담기 3.빼기 4.장바구니 5.쿠폰 6.결제 q.종료")
    print("-" * 46)
    choice = input("선택: ").strip().lower()

    if choice == "1":
        for p in products:
            print(p)

    elif choice == "2":
        number = ask_number("상품 번호: ")
        if number is None:
            continue

        product = find_product(products, number)
        if product is None:
            continue
        if product.is_sold_out():        # 수량 묻기 전에 품절 먼저 확인
            print("품절된 상품입니다.")
            continue
        qty = ask_number("수량: ")
        if qty is None or qty == 0:
            print("1 이상의 수량을 입력해주세요.")
            continue
        cart.add(product, qty)

    elif choice == "3":
        number = ask_number("상품 번호: ")
        if number is None:
            continue
        product = find_product(products, number)
        if product is not None:
            cart.remove(product)

    elif choice == "4":
        if not cart.items:
            print("장바구니가 비어 있습니다.")
        else:
            cart.print_summary()

    elif choice == "5":
        for i, c in enumerate(coupons, start=1):
            print(f"{i}. {c.name}")
        number = ask_number("쿠폰 번호: ")
        if number is None:
            continue

        if 1 <= number <= len(coupons):
            cart.set_coupon(coupons[number - 1])
        else:
            print("없는 쿠폰 번호입니다.")

    elif choice == "6":
        cart.checkout()

    elif choice == "q":
        print("프로그램을 종료합니다.")
        break

    else:
        print("메뉴에서 선택해주세요.")

----------------------------------------------
1.상품목록 2.담기 3.빼기 4.장바구니 5.쿠폰 6.결제 q.종료
----------------------------------------------
보조배터리 3개를 담았습니다.
----------------------------------------------
1.상품목록 2.담기 3.빼기 4.장바구니 5.쿠폰 6.결제 q.종료
----------------------------------------------
숫자만 입력해주세요.
1 이상의 수량을 입력해주세요.
----------------------------------------------
1.상품목록 2.담기 3.빼기 4.장바구니 5.쿠폰 6.결제 q.종료
----------------------------------------------
프로그램을 종료합니다.
